# Professional Process Flow Diagrams (PFDs) from NeqSim

The classic `comparesimulations` notebook renders a process with `process.toDOT()`, which
produces a plain Graphviz graph of labelled boxes. That is fine for quickly checking topology,
but it does **not** look like the PFDs engineers use in industry.

NeqSim also ships a dedicated **professional PFD exporter** — `ProcessDiagramExporter` —
that follows oil & gas drawing conventions (ISO 10628 / ANSI Y32.11):

- **P&ID-style equipment symbols** (cylinders for vessels/columns, trapezoids for
  compressors/expanders, circles for pumps, bow-ties for valves, ...).
- **Phase-aware, colored stream lines** (gas, oil, water, recycle).
- **Gravity / left-to-right layout** (feeds on the left, products on the right; gas up,
  liquids down).
- **Selectable simulator look** — `NEQSIM`, `HYSYS`, `ASPEN_PLUS`, `PROII`.
- **Detail levels** — `CONCEPTUAL`, `ENGINEERING`, `DEBUG`.

This notebook builds a small gas-processing train and compares the basic `toDOT()` output
with the professional exporter in several styles.

## 1. Install and import

We need `neqsim` plus the Python `graphviz` package to render the DOT text inline. On Google
Colab we also install the Graphviz system binaries (`dot`).

In [ ]:
# Colab / fresh environment setup (safe to re-run)
%pip install -q neqsim graphviz
# System Graphviz binaries (needed to RENDER dot -> svg/png). Comment out if already installed.
import shutil, subprocess, sys
if shutil.which('dot') is None:
    try:
        subprocess.run(['apt-get', '-qq', 'install', '-y', 'graphviz'], check=False)
    except Exception as exc:
        print('Could not auto-install Graphviz binaries:', exc)
        print('Install Graphviz manually: https://graphviz.org/download/')

In [ ]:
import jpype
from neqsim import jneqsim
import graphviz
from IPython.display import display

# Java classes
SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
Stream = jneqsim.process.equipment.stream.Stream
ProcessSystem = jneqsim.process.processmodel.ProcessSystem
ThreePhaseSeparator = jneqsim.process.equipment.separator.ThreePhaseSeparator
Cooler = jneqsim.process.equipment.heatexchanger.Cooler
Compressor = jneqsim.process.equipment.compressor.Compressor
Pump = jneqsim.process.equipment.pump.Pump
ThrottlingValve = jneqsim.process.equipment.valve.ThrottlingValve

# Diagram enums
DiagramStyle = jneqsim.process.processmodel.diagram.DiagramStyle
DiagramDetailLevel = jneqsim.process.processmodel.diagram.DiagramDetailLevel


def show_dot(dot_text, caption=None):
    """Render a Graphviz DOT string inline in the notebook."""
    if caption:
        print(caption)
    display(graphviz.Source(dot_text))

## 2. Build a small gas-processing train

Feed gas is cooled, sent to a three-phase separator, the gas is recompressed for export, and
the oil is pumped and let down through a valve. This gives a good variety of equipment symbols.

In [ ]:
fluid = SystemSrkEos(273.15 + 60.0, 60.0)
for comp, frac in [('nitrogen', 0.01), ('CO2', 0.02), ('methane', 0.80),
                   ('ethane', 0.06), ('propane', 0.04), ('n-butane', 0.02),
                   ('n-pentane', 0.01), ('n-hexane', 0.01), ('water', 0.03)]:
    fluid.addComponent(comp, frac)
fluid.setMixingRule('classic')
fluid.setMultiPhaseCheck(True)

feed = Stream('well stream', fluid)
feed.setFlowRate(50.0, 'MSm3/day')
feed.setTemperature(60.0, 'C')
feed.setPressure(60.0, 'bara')

inlet_cooler = Cooler('inlet cooler', feed)
inlet_cooler.setOutTemperature(30.0, 'C')

separator = ThreePhaseSeparator('inlet separator', inlet_cooler.getOutletStream())

export_compressor = Compressor('export compressor', separator.getGasOutStream())
export_compressor.setOutletPressure(120.0, 'bara')

oil_pump = Pump('oil export pump', separator.getOilOutStream())
oil_pump.setOutletPressure(90.0, 'bara')

water_valve = ThrottlingValve('water letdown valve', separator.getWaterOutStream())
water_valve.setOutletPressure(5.0, 'bara')

process = ProcessSystem('Gas Processing Train')
for unit in [feed, inlet_cooler, separator, export_compressor, oil_pump, water_valve]:
    process.add(unit)
process.run()
print('Process converged. Export gas pressure:',
      round(export_compressor.getOutletStream().getPressure('bara'), 1), 'bara')

## 3. Baseline — the plain `toDOT()` (what `comparesimulations` uses)

Functional, but just labelled boxes. Hard to tell a compressor from a separator at a glance.

In [ ]:
basic_dot = process.toDOT()
show_dot(basic_dot, caption='Baseline process.toDOT() output:')

## 4. Professional exporter — HYSYS look

`process.createDiagramExporter()` returns a fluent `ProcessDiagramExporter`. Equipment now uses
P&ID symbols, streams are phase-colored, and the layout flows left-to-right with feeds on the
left and products on the right.

In [ ]:
hysys_dot = (process.createDiagramExporter()
             .setTitle('Gas Processing Train  —  HYSYS style')
             .setDiagramStyle(DiagramStyle.HYSYS)
             .setDetailLevel(DiagramDetailLevel.ENGINEERING)
             .setShowStreamValues(True)
             .setShowLegend(True)
             .toDOT())
show_dot(hysys_dot, caption='Professional PFD (HYSYS style, ENGINEERING detail):')

## 5. Compare simulator looks — Aspen Plus and PRO/II

The same model, different drawing conventions. Aspen Plus uses function-specific colors and
curved streams; PRO/II is a clean black-and-white technical style that prints well.

In [ ]:
aspen_dot = (process.createDiagramExporter()
             .setTitle('Gas Processing Train  —  Aspen Plus style')
             .setDiagramStyle(DiagramStyle.ASPEN_PLUS)
             .setDetailLevel(DiagramDetailLevel.ENGINEERING)
             .toDOT())
show_dot(aspen_dot, caption='Aspen Plus style:')

In [ ]:
proii_dot = (process.createDiagramExporter()
             .setTitle('Gas Processing Train  —  PRO/II style')
             .setDiagramStyle(DiagramStyle.PROII)
             .setDetailLevel(DiagramDetailLevel.ENGINEERING)
             .toDOT())
show_dot(proii_dot, caption='PRO/II (black & white technical) style:')

## 6. Detail levels — CONCEPTUAL vs ENGINEERING

Use `CONCEPTUAL` for a clean teaching / overview diagram (topology only, no process data), and
`ENGINEERING` for a full PFD with temperatures, pressures and flow rates on the streams.

In [ ]:
conceptual_dot = (process.createDiagramExporter()
                  .setTitle('Gas Processing Train  —  Conceptual')
                  .setDiagramStyle(DiagramStyle.NEQSIM)
                  .setDetailLevel(DiagramDetailLevel.CONCEPTUAL)
                  .setShowStreamValues(False)
                  .toDOT())
show_dot(conceptual_dot, caption='Conceptual (clean overview):')

## 7. Export to SVG / PNG / PDF files

For reports and presentations, export directly to image files. These require the Graphviz
`dot` binary (installed in section 1). The exporter writes the file for you.

In [ ]:
from pathlib import Path

exporter = (process.createDiagramExporter()
            .setTitle('Gas Processing Train')
            .setDiagramStyle(DiagramStyle.HYSYS)
            .setDetailLevel(DiagramDetailLevel.ENGINEERING))

out_dir = Path('pfd_output')
out_dir.mkdir(exist_ok=True)

# export* methods take a java.nio.file.Path
JPaths = jpype.JClass('java.nio.file.Paths')
exporter.exportSVG(JPaths.get(str(out_dir / 'gas_train.svg')))
exporter.exportPNG(JPaths.get(str(out_dir / 'gas_train.png')))
print('Wrote:', [p.name for p in out_dir.glob('gas_train.*')])

## Summary

| Goal | Use |
|------|-----|
| Quick topology check | `process.toDOT()` |
| Clean teaching / overview diagram | `createDiagramExporter().setDetailLevel(CONCEPTUAL)` |
| Full engineering PFD | `createDiagramExporter().setDetailLevel(ENGINEERING)` |
| Specific simulator look | `.setDiagramStyle(HYSYS / ASPEN_PLUS / PROII / NEQSIM)` |
| Image files for reports | `.exportSVG(path)` / `.exportPNG(path)` / `.exportPDF(path)` |

For multi-area plant models built as a `ProcessModel`, use `plant.toDOT()` for a plant-wide
view with one cluster per area, or `plant.exportAreaDOT(dir)` to write one readable diagram per
process area.